In [9]:
# Imports and Configuration

import os
import time
import math
import logging
import platform
import psutil
import requests
import wikipediaapi
from datetime import datetime
from dataclasses import dataclass, field
from typing import Optional, List, Dict, Callable, Any
from pathlib import Path
from dotenv import load_dotenv
from groq import Groq
import pandas as pd

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
log = logging.getLogger(__name__)

MAX_RETRIES = 3
RETRY_DELAY = 2
DEFAULT_MODEL = "llama-3.3-70b-versatile"
DEFAULT_MAX_TOKENS = 2048
DEFAULT_TEMPERATURE = 0.3
MAX_TOOL_OUTPUT = 500

In [2]:
# Client Initialization

env_path = Path("C:/educational files/advanced_agent/.env")
load_dotenv(dotenv_path=env_path)

def init_client() -> Groq:
    api_key = os.getenv("GROQ_API_KEY")
    if not api_key:
        raise EnvironmentError("GROQ_API_KEY not found in .env")
    log.info("Groq client initialized successfully")
    return Groq(api_key=api_key)

client = init_client()

2026-06-02 11:40:57,860 [INFO] Groq client initialized successfully


In [3]:
# Agent Configuration

@dataclass
class AgentConfig:
    model: str = DEFAULT_MODEL
    max_tokens: int = DEFAULT_MAX_TOKENS
    temperature: float = DEFAULT_TEMPERATURE
    system_prompt: str = (
        "You are an advanced autonomous AI agent with access to 10 tools. "
        "When a user asks something that requires a tool, respond ONLY with a JSON object in this exact format:\n"
        '{"tool": "tool_name", "input": "tool_input"}\n'
        "Available tools: calculator, wikipedia, web_search, file_reader, datetime_tool, "
        "unit_converter, dictionary, weather, csv_analyzer, system_info.\n"
        "If no tool is needed, respond normally in plain text. "
        "Never mix JSON and plain text in the same response."
    )
    session_token_count: int = field(default=0, repr=False)
    tool_call_count: int = field(default=0, repr=False)

config = AgentConfig()
log.info(f"Agent configured — model: {config.model} | tools: 10")

2026-06-02 11:41:37,482 [INFO] Agent configured — model: llama-3.3-70b-versatile | tools: 10


In [10]:
# Tool Definitions

def calculator(expression: str) -> str:
    try:
        result = eval(expression, {"__builtins__": {}}, {"math": math})
        return f"Result: {result}"
    except Exception as e:
        return f"Calculator error: {e}"

def wikipedia(query: str) -> str:
    try:
        wiki = wikipediaapi.Wikipedia(language="en", user_agent="AutonomousAgent/1.0")
        page = wiki.page(query)
        if not page.exists():
            return f"No Wikipedia page found for: {query}"
        return page.summary[:MAX_TOOL_OUTPUT]
    except Exception as e:
        return f"Wikipedia error: {e}"

def web_search(query: str) -> str:
    try:
        from ddgs import DDGS
        with DDGS() as ddgs:
            results = list(ddgs.text(query, max_results=3))
        if not results:
            return "No results found."
        return "\n".join([f"{r['title']}: {r['body'][:150]}" for r in results])
    except Exception as e:
        return f"Web search error: {e}"

def file_reader(filepath: str) -> str:
    try:
        path = Path(filepath)
        if not path.exists():
            return f"File not found: {filepath}"
        if path.suffix == ".csv":
            df = pd.read_csv(path)
            return f"CSV loaded — shape: {df.shape}\nColumns: {list(df.columns)}\nPreview:\n{df.head(3).to_string()}"
        return path.read_text(encoding="utf-8")[:MAX_TOOL_OUTPUT]
    except Exception as e:
        return f"File reader error: {e}"

def datetime_tool(query: str) -> str:
    try:
        now = datetime.now()
        if "time" in query.lower():
            return f"Current time: {now.strftime('%H:%M:%S')}"
        if "day" in query.lower():
            return f"Today is: {now.strftime('%A')}"
        return f"Current datetime: {now.strftime('%Y-%m-%d %H:%M:%S')}"
    except Exception as e:
        return f"Datetime error: {e}"

def unit_converter(query: str) -> str:
    try:
        parts = query.lower().split()
        value = float(parts[0])
        unit_from = parts[1]
        unit_to = parts[3]
        conversions = {
            ("kg", "lbs"): lambda x: x * 2.20462,
            ("lbs", "kg"): lambda x: x / 2.20462,
            ("km", "miles"): lambda x: x * 0.621371,
            ("miles", "km"): lambda x: x / 0.621371,
            ("celsius", "fahrenheit"): lambda x: x * 9/5 + 32,
            ("fahrenheit", "celsius"): lambda x: (x - 32) * 5/9,
            ("meters", "feet"): lambda x: x * 3.28084,
            ("feet", "meters"): lambda x: x / 3.28084,
        }
        key = (unit_from, unit_to)
        if key not in conversions:
            return f"Conversion from {unit_from} to {unit_to} not supported."
        result = conversions[key](value)
        return f"{value} {unit_from} = {round(result, 4)} {unit_to}"
    except Exception as e:
        return f"Unit converter error: {e}"

def dictionary(word: str) -> str:
    try:
        response = requests.get(f"https://api.dictionaryapi.dev/api/v2/entries/en/{word}", timeout=5)
        data = response.json()
        if isinstance(data, list):
            meaning = data[0]["meanings"][0]
            definition = meaning["definitions"][0]["definition"]
            part_of_speech = meaning["partOfSpeech"]
            return f"{word} ({part_of_speech}): {definition}"
        return f"No definition found for: {word}"
    except Exception as e:
        return f"Dictionary error: {e}"

def weather(city: str) -> str:
    try:
        url = f"https://wttr.in/{city.replace(' ', '+')}?format=3"
        response = requests.get(url, timeout=5)
        return response.text.strip()
    except Exception as e:
        return f"Weather error: {e}"

def csv_analyzer(filepath: str) -> str:
    try:
        df = pd.read_csv(filepath)
        stats = df.describe().to_string()
        return f"Shape: {df.shape}\nColumns: {list(df.columns)}\nStats:\n{stats[:MAX_TOOL_OUTPUT]}"
    except Exception as e:
        return f"CSV analyzer error: {e}"

def system_info(query: str) -> str:
    try:
        cpu = psutil.cpu_percent(interval=1)
        ram = psutil.virtual_memory()
        disk = psutil.disk_usage("/")
        return (
            f"OS: {platform.system()} {platform.release()}\n"
            f"CPU usage: {cpu}%\n"
            f"RAM: {ram.used / 1e9:.1f}GB used / {ram.total / 1e9:.1f}GB total ({ram.percent}%)\n"
            f"Disk: {disk.used / 1e9:.1f}GB used / {disk.total / 1e9:.1f}GB total ({disk.percent}%)"
        )
    except Exception as e:
        return f"System info error: {e}"

log.info("10 tools defined successfully")

C:\ProgramData\anaconda3\Lib\collections\__init__.py:452: ResourceWarning: unclosed <ssl.SSLSocket fd=1620, family=23, type=1, proto=0, laddr=('2406:b400:d5:38bc:a508:bed4:459:5ea4', 63788, 0, 0), raddr=('2001:df2:e500:ed1a::1', 443, 0, 0)>
  result = tuple_new(cls, iterable)
2026-06-02 11:49:56,111 [INFO] 10 tools defined successfully


In [11]:
# Tool Registry

TOOL_REGISTRY: Dict[str, Dict[str, Any]] = {
    "calculator": {
        "fn": calculator,
        "description": "Evaluates math expressions. Input: math expression as string."
    },
    "wikipedia": {
        "fn": wikipedia,
        "description": "Fetches Wikipedia summary. Input: search query."
    },
    "web_search": {
        "fn": web_search,
        "description": "Searches the web via DuckDuckGo. Input: search query."
    },
    "file_reader": {
        "fn": file_reader,
        "description": "Reads txt, csv, or pdf files. Input: full file path."
    },
    "datetime_tool": {
        "fn": datetime_tool,
        "description": "Returns current date, time, or day. Input: query like 'what time is it'."
    },
    "unit_converter": {
        "fn": unit_converter,
        "description": "Converts units. Input: '100 km to miles' format."
    },
    "dictionary": {
        "fn": dictionary,
        "description": "Returns word definition. Input: single word."
    },
    "weather": {
        "fn": weather,
        "description": "Returns current weather. Input: city name."
    },
    "csv_analyzer": {
        "fn": csv_analyzer,
        "description": "Analyzes CSV file statistics. Input: full file path."
    },
    "system_info": {
        "fn": system_info,
        "description": "Returns system CPU, RAM, disk usage. Input: any string."
    }
}

log.info(f"Tool registry ready — {len(TOOL_REGISTRY)} tools registered")

2026-06-02 11:50:04,903 [INFO] Tool registry ready — 10 tools registered


In [12]:
# Tool Executor

def execute_tool(tool_name: str, tool_input: str) -> str:
    if tool_name not in TOOL_REGISTRY:
        return f"Unknown tool: {tool_name}"
    try:
        log.info(f"Executing tool: {tool_name} | input: {tool_input[:80]}")
        result = TOOL_REGISTRY[tool_name]["fn"](tool_input)
        log.info(f"Tool {tool_name} completed successfully")
        return result
    except Exception as e:
        log.error(f"Tool {tool_name} failed: {e}")
        return f"Tool execution error: {e}"

In [13]:
# Chat Engine with Tool Calling

import json

def chat(user_input: str, cfg: AgentConfig) -> str:
    messages = [
        {"role": "system", "content": cfg.system_prompt},
        {"role": "user", "content": user_input}
    ]

    for attempt in range(MAX_RETRIES):
        try:
            response = client.chat.completions.create(
                model=cfg.model,
                messages=messages,
                max_tokens=cfg.max_tokens,
                temperature=cfg.temperature
            )
            reply = response.choices[0].message.content.strip()
            cfg.session_token_count += response.usage.total_tokens

            try:
                tool_call = json.loads(reply)
                if "tool" in tool_call and "input" in tool_call:
                    tool_name = tool_call["tool"]
                    tool_input = tool_call["input"]
                    cfg.tool_call_count += 1
                    log.info(f"Tool call #{cfg.tool_call_count}: {tool_name}")
                    tool_result = execute_tool(tool_name, tool_input)

                    messages.append({"role": "assistant", "content": reply})
                    messages.append({"role": "user", "content": f"Tool result: {tool_result}"})

                    final_response = client.chat.completions.create(
                        model=cfg.model,
                        messages=messages,
                        max_tokens=cfg.max_tokens,
                        temperature=cfg.temperature
                    )
                    final_reply = final_response.choices[0].message.content.strip()
                    cfg.session_token_count += final_response.usage.total_tokens
                    log.info(f"Tokens session total: {cfg.session_token_count}")
                    return final_reply

            except (json.JSONDecodeError, KeyError):
                log.info(f"Tokens session total: {cfg.session_token_count}")
                return reply

        except Exception as e:
            log.warning(f"Attempt {attempt + 1} failed: {e}")
            if attempt < MAX_RETRIES - 1:
                time.sleep(RETRY_DELAY)

    log.error("All retry attempts failed")
    return "Agent failed to respond."

In [14]:
# Interactive Chat Loop

print("Agent ready. Commands: 'exit' to quit | 'tools' to list tools | 'stats' for usage\n")

while True:
    user_input = input("You: ").strip()

    if not user_input:
        continue
    if user_input.lower() == "exit":
        print(f"Session ended. Tokens: {config.session_token_count} | Tool calls: {config.tool_call_count}")
        break
    if user_input.lower() == "tools":
        print("\nAvailable tools:")
        for name, meta in TOOL_REGISTRY.items():
            print(f"  {name}: {meta['description']}")
        print()
        continue
    if user_input.lower() == "stats":
        print(f"Tokens used: {config.session_token_count} | Tool calls: {config.tool_call_count}\n")
        continue

    reply = chat(user_input, config)
    print(f"\nAgent: {reply}\n")

Agent ready. Commands: 'exit' to quit | 'tools' to list tools | 'stats' for usage



You:  latest AI news


2026-06-02 11:50:27,055 [INFO] HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-06-02 11:50:27,060 [INFO] Tool call #9: web_search
2026-06-02 11:50:27,062 [INFO] Executing tool: web_search | input: latest AI news
2026-06-02 11:50:28,116 [INFO] response: https://en.wikipedia.org/w/api.php?action=opensearch&profile=fuzzy&limit=1&search=latest%20AI%20news 200
2026-06-02 11:50:30,314 [INFO] response: https://grokipedia.com/api/typeahead?query=latest+AI+news&limit=1 200
2026-06-02 11:50:31,651 [INFO] response: https://www.google.com/search?q=latest+AI+news&filter=1&start=0&hl=en-US&lr=lang_en&cr=countryUS 200
2026-06-02 11:50:32,226 [INFO] Tool web_search completed successfully
2026-06-02 11:50:32,884 [INFO] HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-06-02 11:50:32,893 [INFO] Tokens session total: 3599



Agent: It appears that there have been several recent developments in the field of AI. Some of the latest news includes the failure of AI coding agents to work effectively in teamwork, the potential for AI hiring tools to perpetuate racial bias, and updates on the latest research and advancements from organizations such as OpenAI and Meta. Additionally, there are reports of issues with OpenAI's ChatGPT platform. For more information, you can check out the latest news and updates from reputable sources such as Stanford HAI, AI Magazine, and other online publications.



You:  exit


Session ended. Tokens: 3599 | Tool calls: 9
